# bagpipe DB review

Exploratory only — no outputs committed (nbstripout). Reads the shared
brainlink.db via bagpipe's config, no paths/data hardcoded here.

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bagpipe.core.config import get_path

sns.set_theme(style="whitegrid")
con = sqlite3.connect(get_path("db_path"))

## 1. Table overview

In [ ]:
tables = pd.read_sql("select name from sqlite_master where type='table'", con)["name"]
counts = {t: pd.read_sql(f"select count(*) c from {t}", con)["c"][0] for t in tables}
pd.Series(counts, name="rows").sort_values(ascending=False).to_frame()

## 2. SNBB cohort (S####)

In [ ]:
snbb = pd.read_sql('''
    select s.session_id, s.uid, s.lab, s.scan_date, s.mapping_complete,
           d.age_at_scan, d.sex
    from session s
    left join demographics d on d.session_id = s.session_id
    where s.uid is not null
''', con, parse_dates=["scan_date"])
print(len(snbb), "sessions,", snbb["uid"].nunique(), "subjects")
snbb.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
snbb["age_at_scan"].dropna().hist(bins=40, ax=axes[0])
axes[0].set_title("Age at scan (SNBB)")
snbb["lab"].value_counts().plot.bar(ax=axes[1])
axes[1].set_title("Sessions per lab")
plt.tight_layout()

In [ ]:
snbb.set_index("scan_date").resample("QE").size().plot(figsize=(9, 3), title="SNBB sessions per quarter")
plt.tight_layout()

In [ ]:
missing = pd.Series({
    "no age": snbb["age_at_scan"].isna().mean(),
    "no sex": snbb["sex"].isna().mean(),
    "mapping incomplete": (~snbb["mapping_complete"].astype(bool)).mean(),
})
(missing * 100).round(1).to_frame("pct missing")

## 3. Legacy cohort (pre-SNBB, 12-digit ids)

In [ ]:
legacy = pd.read_sql("select * from legacy_participant", con, parse_dates=["scan_date"])
print(len(legacy), "rows")
legacy.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
legacy["age_at_scan"].dropna().hist(bins=40, ax=axes[0])
axes[0].set_title("Age at scan (legacy)")
legacy["lab"].value_counts().plot.bar(ax=axes[1])
axes[1].set_title("Sessions per lab (legacy)")
plt.tight_layout()

## 4. CAT12 tabular features (`features` table)

Long format: one row per (subject, session, atlas, region, metric).

In [ ]:
feat_summary = pd.read_sql('''
    select source, atlas, metric, count(*) n, avg(value) mean_value
    from features group by source, atlas, metric order by n desc
''', con)
feat_summary

In [ ]:
globals_df = pd.read_sql('''
    select subject_key, session_id, metric, value
    from features where region = 'global'
''', con)
globals_wide = globals_df.pivot_table(index=["subject_key", "session_id"], columns="metric", values="value")
globals_wide.describe()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, ["TIV", "vol_gm", "vol_wm", "vol_csf"]):
    globals_wide[col].hist(bins=50, ax=ax)
    ax.set_title(col)
plt.tight_layout()

### Age vs. global GM volume — sanity check for a brain-age signal

In [ ]:
age_snbb = snbb.set_index(["uid", "session_id"])["age_at_scan"]
age_legacy = legacy.set_index("subject_id")["age_at_scan"]

gm = globals_wide["vol_gm"].reset_index()
gm["age"] = gm.apply(
    lambda r: age_snbb.get((r["subject_key"], r["session_id"]), age_legacy.get(r["subject_key"])),
    axis=1,
)
gm = gm.dropna(subset=["age"])
plt.figure(figsize=(6, 4))
sns.regplot(data=gm, x="age", y="vol_gm", scatter_kws={"s": 8, "alpha": 0.3}, line_kws={"color": "red"})
plt.title(f"GM volume vs age (n={len(gm)}); corr={gm['age'].corr(gm['vol_gm']):.2f}")
plt.tight_layout()

## 5. Imaging paths (for DL training)

In [ ]:
img_snbb = pd.read_sql("select processing_stream, count(*) n from imaging_path group by processing_stream", con)
img_legacy = pd.read_sql("select processing_stream, count(*) n from legacy_imaging_path group by processing_stream", con)
print("SNBB:\n", img_snbb, "\n\nLegacy:\n", img_legacy)

In [ ]:
sizes = pd.read_sql("select processing_stream, file_size_bytes from imaging_path", con)
sizes["file_size_mb"] = sizes["file_size_bytes"] / 1e6
sns.boxplot(data=sizes, x="processing_stream", y="file_size_mb")
plt.title("SNBB image file size by stream (sanity check — should be tight)")
plt.tight_layout()

## 6. Coverage: sessions with features vs. images vs. demographics

In [ ]:
cov = pd.read_sql('''
    select s.uid, s.session_id,
           d.session_id is not null as has_demo,
           f.session_id is not null as has_features,
           ip.session_id is not null as has_image
    from session s
    left join demographics d on d.session_id = s.session_id
    left join (select distinct session_id from features) f on f.session_id = s.session_id
    left join (select distinct session_id from imaging_path) ip on ip.session_id = s.session_id
    where s.uid is not null
''', con)
cov[["has_demo", "has_features", "has_image"]].mean().mul(100).round(1).to_frame("pct of SNBB sessions")

In [ ]:
combo = cov.groupby(["has_demo", "has_features", "has_image"]).size().rename("n").reset_index()
combo["combo"] = combo.apply(
    lambda r: "+".join(
        n for n, has in zip(["demo", "features", "image"], [r.has_demo, r.has_features, r.has_image]) if has
    ) or "none", axis=1,
)
combo.set_index("combo")["n"].sort_values().plot.barh(figsize=(7, 3), title="SNBB session coverage combinations")
plt.tight_layout()

## 7. Validation issues logged by brainlink ingestion

In [ ]:
val = pd.read_sql("select level, entity_type, check_name, count(*) n from validation_log group by 1,2,3 order by n desc limit 30", con)
val

## 8. Training tables (`bag export training-table`)

Three Parquet tables, all keyed by (subject_key, session_id):
- `globals.parquet` — wide, tabular model input
- `regional.parquet` — long, per-region base-learner input (stacked ensemble)
- `image_paths.parquet` — CNN input

In [ ]:
from bagpipe.core.config import get_path

ds_dir = get_path("datasets_dir")
globals_df = pd.read_parquet(ds_dir / "globals.parquet")
regional_df = pd.read_parquet(ds_dir / "regional.parquet")
paths_df = pd.read_parquet(ds_dir / "image_paths.parquet")
for name, df in [("globals", globals_df), ("regional", regional_df), ("image_paths", paths_df)]:
    print(name, df.shape)

### 8.1 Globals

In [ ]:
globals_df.describe(include="all")

In [ ]:
globals_df.isna().mean().mul(100).round(1).to_frame("pct missing")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
globals_df["cohort"].value_counts().plot.bar(ax=axes[0], title="Rows per cohort")
globals_df.groupby("cohort")["age"].plot.hist(bins=40, alpha=0.6, ax=axes[1], legend=True)
axes[1].set_title("Age distribution by cohort")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
for ax, col in zip(axes, ["TIV", "vol_gm", "vol_wm", "vol_csf"]):
    sns.boxplot(data=globals_df, x="cohort", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()

Outlier check — TIV should never be near zero:

In [ ]:
globals_df.nsmallest(10, "TIV")[["subject_key", "session_id", "cohort", "age", "TIV", "vol_gm"]]

In [ ]:
sns.pairplot(
    globals_df[["age", "TIV", "vol_gm", "vol_wm", "vol_csf", "cohort"]].dropna(),
    hue="cohort", plot_kws={"s": 8, "alpha": 0.3}, diag_kind="hist",
)

### 8.2 Regional (long)

In [ ]:
print(regional_df[["atlas", "region", "metric"]].nunique())
regional_df.groupby("metric")["value"].describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, metric in zip(axes, regional_df["metric"].unique()):
    regional_df.loc[regional_df.metric == metric, "value"].hist(bins=60, ax=ax)
    ax.set_title(metric)
plt.tight_layout()

In [ ]:
region_means = (
    regional_df[regional_df.metric == "vol_gm"]
    .groupby("region")["value"].mean().sort_values(ascending=False)
)
region_means.head(20).plot.barh(figsize=(6, 6), title="Top 20 regions by mean GM volume (ml)")
plt.gca().invert_yaxis()
plt.tight_layout()

In [ ]:
sessions_per_region = regional_df.groupby("region").size()
print("regions:", len(sessions_per_region), "- rows/region range:", sessions_per_region.min(), "-", sessions_per_region.max())

### 8.3 Image paths

In [ ]:
paths_df["cohort"].value_counts()

In [ ]:
paths_df[["image_path_mwp1", "image_path_wm"]].notna().mean().mul(100).round(1).to_frame("pct present")

### 8.4 Coverage across all three tables

In [ ]:
key = globals_df[["subject_key", "session_id"]]
has_regional = key.merge(regional_df[["subject_key", "session_id"]].drop_duplicates(), how="left", indicator=True)["_merge"].eq("both")
has_image = key.merge(paths_df[["subject_key", "session_id"]], how="left", indicator=True)["_merge"].eq("both")
pd.Series({
    "globals row has regional data": has_regional.mean(),
    "globals row has image path": has_image.mean(),
}).mul(100).round(1).to_frame("pct")